# 第4章 库、脚本与 Jupyter

这个 notebook 不只是概念说明，而是一个最小可运行工作流：检查标准库和第三方库、写一个小脚本、在 notebook 里调用它、并记录随机种子和环境信息。重点是理解这些工具在科研流程中的分工，而不是把它们混成同一种东西。

## 学习目标

- 理解库、模块、脚本和 notebook 在工作流中的不同角色
- 学会检查当前环境里哪些依赖可用
- 看到 notebook 和脚本如何配合而不是互相替代
- 认识 `main()`、配置和日志如何让脚本更稳定
- 建立记录随机种子与环境快照的基本习惯

In [ ]:
from __future__ import annotations

import platform
import random
import statistics
import subprocess
import tempfile
from importlib.util import find_spec
from pathlib import Path

print('Python version:', platform.python_version())
print('Platform:', platform.platform())


## 标准库与第三方库

标准库随 Python 自带；第三方库需要额外安装。科研工作里，先确认环境状况是一种很重要的习惯。

In [ ]:
optional_modules = ['numpy', 'matplotlib', 'astropy', 'pandas', 'scikit-learn']
availability = {module_name: (find_spec(module_name) is not None) for module_name in optional_modules}
print('Optional module availability:')
for name, available in availability.items():
    print(f'  {name}: {available}')


## 在 notebook 里做快速探索

notebook 最适合先试验、先看结果、先解释。下面这个单元就是典型的探索型片段。

In [ ]:
import math

random.seed(42)
values = [x**2 + math.sin(3 * x) for x in range(5)]
noise = [round(random.random(), 4) for _ in range(3)]

print('values =', values)
print('mean =', round(statistics.mean(values), 3))
print('noise sample =', noise)


## 当逻辑要重复时，就该整理成脚本

很多科研代码最开始都诞生于 notebook，但一旦某段逻辑需要重复跑、批量跑或者交给别人运行，就应该开始脚本化。

In [ ]:
workspace = tempfile.TemporaryDirectory(prefix='aifor_scripts_demo_')
WORKDIR = Path(workspace.name)
SCRIPT_PATH = WORKDIR / 'compute_values.py'

script_text = '''import math
for x in range(5):
    print(x, x**2 + math.sin(3*x))
'''
SCRIPT_PATH.write_text(script_text, encoding='utf-8')

print('temporary script path:', SCRIPT_PATH)
print('script contents:')
print(SCRIPT_PATH.read_text(encoding='utf-8'))


## 在 notebook 里调用脚本

这是一种很常见的科研配合方式：notebook 负责讲解和组织结果，脚本负责稳定、重复地执行。

In [ ]:
completed = subprocess.run(
    ['python', str(SCRIPT_PATH)],
    capture_output=True,
    text=True,
    check=True,
)
print('script output:')
print(completed.stdout.strip())


## 脚本入口、配置和日志

当脚本开始变成工作流入口时，最好显式写出 `main()`、配置对象和最小日志。这样 notebook 可以调用脚本，脚本也能独立运行。

In [ ]:
catalog_path = WORKDIR / 'tiny_catalog.csv'
catalog_path.write_text('object_id,value\nA,1.0\nB,2.5\nC,4.0\n', encoding='utf-8')
ENTRY_SCRIPT_PATH = WORKDIR / 'summarize_catalog.py'
summary_path = WORKDIR / 'summary.txt'

entry_script = f'''import csv
from pathlib import Path

CONFIG = {{
    "input_path": r"{catalog_path}",
    "output_path": r"{summary_path}",
}}


def main(config):
    input_path = Path(config["input_path"])
    output_path = Path(config["output_path"])
    print(f"Reading {{input_path.name}}")
    with input_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    values = [float(row["value"]) for row in rows]
    output_path.write_text(f"n={{len(values)}}\\nmean={{sum(values) / len(values):.3f}}\\n", encoding="utf-8")
    print(f"Writing {{output_path.name}}")


if __name__ == "__main__":
    main(CONFIG)
'''

ENTRY_SCRIPT_PATH.write_text(entry_script, encoding='utf-8')
entry_completed = subprocess.run(
    ['python', str(ENTRY_SCRIPT_PATH)],
    capture_output=True,
    text=True,
    check=True,
)
print('entry script log:')
print(entry_completed.stdout.strip())
print('summary file:')
print(summary_path.read_text(encoding='utf-8').strip())


## 可复现性：随机种子和环境快照

如果你的分析包含随机过程，必须记录种子；如果依赖环境差异明显，也应至少记录版本和库可用性。

In [ ]:
seed = 123
random.seed(seed)
sample_a = [round(random.random(), 5) for _ in range(4)]
random.seed(seed)
sample_b = [round(random.random(), 5) for _ in range(4)]

print('sample_a =', sample_a)
print('sample_b =', sample_b)
print('identical after reseeding =', sample_a == sample_b)


## notebook 与脚本的工作边界

- notebook 适合探索、讲解、可视化和课堂演示。
- 脚本适合批处理、重复运行和自动化任务。
- 库负责把可复用逻辑封装起来，让 notebook 和脚本都能调用。
- 当 notebook 中某段逻辑反复出现时，就应该考虑整理成函数、模块或脚本。

## 小结

这一章最重要的不是会不会某个命令，而是理解工作流边界：环境检查属于准备层，notebook 属于探索和讲解层，脚本属于稳定执行层，而库则负责复用。把这几层分清楚，后面的科研项目才更容易维护和复现。

In [ ]:
snapshot = {
    'workdir': str(WORKDIR),
    'script_exists': SCRIPT_PATH.exists(),
    'entry_script_exists': ENTRY_SCRIPT_PATH.exists(),
    'summary_output_exists': summary_path.exists(),
    'numpy_available': availability['numpy'],
    'matplotlib_available': availability['matplotlib'],
    'python_version': platform.python_version(),
}

print('Libraries/scripts/Jupyter snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')


## V2 Evidence Record artifact

最后把本章 workflow 收束成一个轻量 Evidence Record。注意：示例脚本是在临时工作目录中生成的，持久证据是下面写出的 markdown 文件。


In [ ]:
artifact_dir = Path('results')
artifact_dir.mkdir(parents=True, exist_ok=True)
evidence_path = artifact_dir / 'ch04_evidence_record.md'

module_status = ', '.join(
    f'{name}={"available" if available else "not available"}'
    for name, available in availability.items()
)

record_text = f"""# Evidence Record - Ch04 Libraries, Scripts, and Jupyter

## Record Type

- Record type: preprocessing / reproducibility evidence

## 1. Input

- Data / file path, preferably relative to project root: temporary `tiny_catalog.csv` generated during notebook run
- Data source or generation method: notebook-created teaching CSV with three rows
- Script / notebook: `ch04_libraries_scripts_jupyter.ipynb`; temporary scripts `{SCRIPT_PATH.name}` and `{ENTRY_SCRIPT_PATH.name}`
- Code version / tag, if relevant: fill in the current Git commit or release tag when submitting

## 2. Operation

- Command or entry point: `python {ENTRY_SCRIPT_PATH.name}` inside the temporary work directory
- Key parameters: input path `{catalog_path.name}`; output path `{summary_path.name}`
- Random seed, if relevant: `{seed}`; reseeding check returned `{sample_a == sample_b}`
- Output directory: temporary work directory `{WORKDIR}`; persistent record directory `results/`

## 3. Output

- Output file(s): temporary `{summary_path.name}`; persistent `{evidence_path}`
- Short result summary: summary output exists = `{summary_path.exists()}`; script output exists = `{SCRIPT_PATH.exists() and ENTRY_SCRIPT_PATH.exists()}`
- Generated by which script/notebook: this notebook plus temporary command-line script entry

## 4. Check

- Check performed: script ran with `check=True`; output file existence checked; random seed was rerun twice; package availability recorded
- Evidence from the check: summary output exists = `{summary_path.exists()}`; identical random samples = `{sample_a == sample_b}`; Python `{platform.python_version()}`; packages `{module_status}`

## 5. Limit

- Known limitation: temporary script paths are not durable after the notebook session; copy the script into a project `scripts/` directory for a real submission
- Selection / quality / uncertainty issue, if relevant: teaching data are synthetic and too small for scientific claims

## 6. Reuse in ML

- Sample / feature / label: not a model dataset yet
- Uncertainty / quality flag: not applicable
- Preprocessing record: demonstrates how a notebook hands stable work to a script entry point
- Reproducibility evidence: records environment, seed, command entry, and output existence

## Ch04 Fields

- notebook: `ch04_libraries_scripts_jupyter.ipynb`
- script entry: `{ENTRY_SCRIPT_PATH.name}`
- command to run: `python {ENTRY_SCRIPT_PATH.name}`
- python version: `{platform.python_version()}`
- key package versions: availability only in this minimal example; {module_status}
- random seed: `{seed}`
- output directory: `results/` for persistent record; temporary work directory for demo outputs
"""

evidence_path.write_text(record_text, encoding='utf-8')
print('wrote Evidence Record:', evidence_path)
